# 🎮 梦幻西游抓鬼任务 - YOLOv8 NPC 识别模型训练
## （和鲸社区专用版本）

### 数据集信息
- **图片数量**: 97 张标注图片
- **类别数量**: 7 个
- **类别列表**: 马副将、驿站老板、黑无常、钟馗、主鬼、小怪、鼠标指针

### 预计训练时间
- **GPU**: 5-10 分钟
- **CPU**: 不推荐使用

### 1️⃣ 检查 GPU 环境

In [ ]:
import torch
print(f"PyTorch 版本：{torch.__version__}")
print(f"CUDA 版本：{torch.version.cuda}")
print(f"GPU 可用：{torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 型号：{torch.cuda.get_device_name(0)}")
    print(f"GPU 内存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ 未检测到 GPU，将使用 CPU 训练（较慢）")

### 2️⃣ 安装依赖

In [ ]:
!pip install ultralytics -q
print("✅ 依赖安装完成")

### 3️⃣ 解压数据集

In [ ]:
import zipfile
import os

# 解压标注数据
if os.path.exists('annotation_100.zip'):
    print("📂 正在解压 annotation_100.zip...")
    with zipfile.ZipFile('annotation_100.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print("✅ 解压完成")
else:
    print("❌ 未找到 annotation_100.zip，请先上传文件")

### 4️⃣ 创建数据配置文件

In [ ]:
data_yaml = """
path: dataset/annotation_100
train: images
val: images

names:
  - 马副将
  - 驿站老板
  - 黑无常
  - 钟馗
  - 主鬼
  - 小怪
  - 鼠标指针
"""

with open('data.yaml', 'w', encoding='utf-8') as f:
    f.write(data_yaml)

print("✅ 数据配置完成")
print("📄 配置文件：data.yaml")

### 5️⃣ 开始训练 🚀

In [ ]:
from ultralytics import YOLO
import time

start_time = time.time()

print("="*60)
print("开始训练 YOLOv8 - 抓鬼任务 NPC 识别")
print("="*60)

# 加载预训练模型
print("📥 加载 YOLOv8n 预训练模型...")
model = YOLO('yolov8n.pt')

# 开始训练
print("🚀 开始训练...")
print("配置：100 epochs, batch=16, imgsz=640\n")

results = model.train(
    data='data.yaml',
    epochs=100,      # 训练 100 轮
    batch=16,        # 批次大小
    imgsz=640,       # 图片大小
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    project='trained_model',
    name='zhuagui_v1',
    verbose=True,
    plots=True
)

elapsed = (time.time() - start_time) / 60
print(f"\n✅ 训练完成！用时：{elapsed:.1f} 分钟")

### 6️⃣ 验证模型性能

In [ ]:
print("📊 验证模型性能...")
metrics = model.val()

print("\n" + "="*60)
print("📈 训练结果")
print("="*60)
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"精确率：    {metrics.box.mp:.4f}")
print(f"召回率：    {metrics.box.mr:.4f}")
print("="*60)

if metrics.box.map50 > 0.7:
    print("🎉 模型性能优秀！")
elif metrics.box.map50 > 0.5:
    print("✅ 模型性能良好")
else:
    print("⚠️ 建议增加训练数据或调整参数")

### 7️⃣ 导出模型

In [ ]:
print("📦 导出模型...")

# 导出为 ONNX 格式（推荐用于部署）
onnx_path = model.export(format='onnx')
print(f"✅ ONNX 模型：{onnx_path}")

# 导出为 TorchScript 格式
torchscript_path = model.export(format='torchscript')
print(f"✅ TorchScript 模型：{torchscript_path}")

print("\n✅ 模型导出完成！")

### 8️⃣ 查看训练文件

In [ ]:
import os

print("📁 训练完成的文件:")
print("="*60)

model_dir = 'trained_model/zhuagui_v1/weights'

if os.path.exists(model_dir):
    for file in os.listdir(model_dir):
        file_path = os.path.join(model_dir, file)
        file_size = os.path.getsize(file_path) / 1024 / 1024  # MB
        print(f"  📄 {file}: {file_size:.2f} MB")
else:
    print("❌ 未找到模型目录")

### 9️⃣ 下载模型

In [ ]:
from heywhale import files  # 和鲸社区的文件下载工具

print("📥 准备下载模型...")

# 模型路径
model_path = 'trained_model/zhuagui_v1/weights/best.pt'
onnx_path = 'trained_model/zhuagui_v1/weights/best.onnx'

if os.path.exists(model_path):
    print(f"📦 PyTorch 模型：{model_path}")
    files.download(model_path)
    
    if os.path.exists(onnx_path):
        print(f"📦 ONNX 模型：{onnx_path}")
        files.download(onnx_path)
    
    print("\n✅ 下载完成！")
    print("\n💡 提示：在项目文件浏览器中也可以右键下载文件")
else:
    print("❌ 模型文件未找到")

### 🔟 测试模型（可选）

In [ ]:
# 如果有测试图片，可以上传并测试
print("📤 上传测试图片（可选）...")
print("提示：跳过此步骤直接下载模型即可")

# 示例：使用训练好的模型进行预测
# results = model('test_image.jpg')
# results[0].show()

## 🎉 完成！

### 下一步操作：
1. **下载模型文件**：
   - `best.pt` (PyTorch 格式，约 6MB)
   - `best.onnx` (ONNX 格式，用于部署)

2. **将模型集成到抓鬼自动化系统**

3. **测试 NPC 识别效果**

### 模型保存位置：
`trained_model/zhuagui_v1/weights/best.pt`

### 性能指标：
- mAP50: 0.7-0.85（预期）
- 推理速度：~30 FPS (GPU)
- 模型大小：~6MB